Clean up

In [ ]:
import pandas as pd

# NA
na_cleaned_df = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/NA/EKP/CV/na_cleaned.csv"
)
na_df = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/NA/EKP/parsed/exisiting_EKP_nan_parsed.csv"
)
filtered_na_df = na_df[na_df["Sample_ID_IfH"].isin(na_cleaned_df["Sample_ID_IfH"])]
filtered_na_df.to_csv("EKP_nan_parsed_cleaned.csv", index=False)
print("NA len:")
print(len(filtered_na_df))

# VITEK
vitek_cleaned_df = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/NA/EKP/CV/VITEK_cleaned.csv"
)
vitek_df = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/parsed/VITEK_combined_02_26_parsed.csv"
)
filtered_vitek_df = vitek_df[
    vitek_df["Sample_ID_IfH"].isin(vitek_cleaned_df["Sample_ID_IfH"])
]
filtered_vitek_df.to_csv("EKP_vitek_parsed_cleaned.csv", index=False)
print("VITEK len:")
print(len(filtered_vitek_df))

# Phoenix
phoenix_cleaned_df = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/NA/EKP/CV/phoenix_cleaned.csv"
)
phoenix_df = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/parsed/phoenix_combined_02_26_parsed.csv"
)
filtered_phoenix_df = phoenix_df[
    phoenix_df["Sample_ID_IfH"].isin(phoenix_cleaned_df["Sample_ID_IfH"])
]
filtered_phoenix_df.to_csv("EKP_phoenix_parsed_cleaned.csv", index=False)

print("Phoenix len:")
print(len(filtered_phoenix_df))

/local/tmp/ipykernel_538810/1916596210.py:7: DtypeWarning: Columns (0: imipenem/relebactam, 1: ceftaroline, 2: doxycycline, 3: cephalothin, 4: cefiderocol, 5: florfenicol, 6: sulfamethoxazole) have mixed types. Specify dtype option on import or set low_memory=False.
  na_df = pd.read_csv(


NA len
1862
VITEK len
848
Phoenix len
1575


Train classifier with Phoenix and VITEK EKP Data

In [26]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OrdinalEncoder
from sklearn.neighbors import RadiusNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

df_vitek = pd.read_csv("EKP_vitek_parsed_cleaned.csv")
df_phoenix = pd.read_csv("EKP_phoenix_parsed_cleaned.csv")

df_vitek = df_vitek[df_vitek["Organism_Code"].str.contains("EKP", case=False, na=False)]

df_phoenix = df_phoenix[
    df_phoenix["Organism_Code"].str.contains("EKP", case=False, na=False)
]

df_vitek["device"] = "VITEK"
df_phoenix["device"] = "PHOENIX"
df = pd.concat([df_vitek, df_phoenix], ignore_index=True)
print("Combined shape:", df.shape)

# Clean up
y = df["device"]
exclude_cols = ["Sample_ID_IfH", "device", "Organism_Code"]
X = df.drop(columns=exclude_cols)
X = X.fillna("MISSING")
X = X.astype(str)

# Encode
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_encoded = encoder.fit_transform(X)
X_encoded = pd.DataFrame(encoder.fit_transform(X), columns=X.columns, index=X.index)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

# Train and predict
clf = RadiusNeighborsClassifier(
    radius=0.1,
    metric="hamming",
    weights="distance",
    outlier_label="UNKNOWN",
)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

# Results
report_dict = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv("classification_report.csv", index=True)

print("\nSaved:")
print("classification_report.csv")
labels = sorted(list(set(y_test) | set(y_pred)))
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
cm_df.to_csv("confusion_matrix.csv", index=True)

Combined shape: (2423, 52)

Saved:
classification_report.csv


/projects/envs/conda/jzander/envs/python_notebook/lib/python3.14/site-packages/sklearn/neighbors/_classification.py:868: UserWarning: Outlier label UNKNOWN is not in training classes. All class probabilities of outliers will be assigned with 0.
  warnings.warn(
/projects/envs/conda/jzander/envs/python_notebook/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/projects/envs/conda/jzander/envs/python_notebook/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/projects/envs/conda/jzander/envs/

Get device specific features

In [27]:
results = []

for col in X.columns:

    counts = pd.crosstab(df["device"], X[col])
    rel_counts = pd.crosstab(df["device"], X[col], normalize="index")

    for category in counts.columns:
        row = {"feature": col, "category": category}
        freqs = []
        for device in counts.index:

            count = counts.loc[device, category]
            freq = rel_counts.loc[device, category]
            row[f"{device}_count"] = count
            row[f"{device}_freq"] = round(freq, 5)

            freqs.append(freq)
            
        row["specificity"] = round(max(freqs) - min(freqs), 5)
        results.append(row)

# Save results
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("specificity", ascending=False)
results_df.to_csv("feature_wise_device_specificity.csv", index=False)
print(results_df.head(20))

                           feature category  PHOENIX_count  PHOENIX_freq  \
143                     ampicillin      >16           1551       0.98476   
56                       ertapenem  MISSING           1575       1.00000   
402                      cefazolin      >16           1411       0.89587   
350                     cefuroxime      >16           1404       0.89143   
381           ampicillin/sulbactam    >16/8           1386       0.88000   
145                     ampicillin     >=32              0       0.00000   
190                    ceftriaxone      >32           1361       0.86413   
263                  ciprofloxacin       >2           1352       0.85841   
117                     cefotaxime  MISSING           1575       1.00000   
294                    ceftazidime      >16           1315       0.83492   
82                     tigecycline  MISSING           1575       1.00000   
313                      aztreonam      >16           1245       0.79048   
423         

Prediction function

In [28]:
def predict(df_unknown):
    df_unknown = df_unknown.dropna(subset=df_unknown.columns[2:], how="all")
    df_unknown = df_unknown[
        df_unknown["Organism_Code"].str.contains("EKP", case=False, na=False)
    ]

    # Encode unknown samples
    X_unknown = df_unknown.drop(columns=["Sample_ID_IfH"])
    missing_cols = set(X.columns) - set(X_unknown.columns)
    for col in missing_cols:
        X_unknown[col] = "MISSING"
    extra_cols = set(X_unknown.columns) - set(X.columns)
    if len(extra_cols) > 0:
        X_unknown = X_unknown.drop(columns=list(extra_cols))
    X_unknown = X_unknown[X.columns]
    X_unknown = X_unknown.fillna("MISSING")
    X_unknown = X_unknown.astype(str)
    print(len(X_unknown))
    X_unknown_encoded = encoder.transform(X_unknown)

    # Prediction
    predictions = clf.predict(X_unknown_encoded)
    distances, indices = clf.radius_neighbors(X_unknown_encoded, return_distance=True)

    # Results
    sample_ids = df_unknown["Sample_ID_IfH"]
    results = pd.DataFrame(
        {
            "Sample_ID": sample_ids,
            "Predicted_Device": predictions,
            "Neighbor_Count": [len(i) for i in indices],
            "Mean_Distance": [np.mean(d) if len(d) > 0 else np.nan for d in distances],
        }
    )

    print("\nPredicted Device Distribution")
    print(results["Predicted_Device"].value_counts(normalize=True))

Predict MicroScan and Sensititre to validate UNKNOWN classification

In [29]:
df_microscan = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/parsed/MicroScan_combined_02_26_parsed.csv"
)
df_sensititre = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/parsed/Sensititre_combined_02_26_parsed.csv"
)
print("Microscan Prediction:")
predict(df_microscan)
print("Sensititre Prediction:")
predict(df_sensititre)

Microscan Prediction:
23

Predicted Device Distribution
Predicted_Device
UNKNOWN    1.0
Name: proportion, dtype: float64
Sensititre Prediction:
22

Predicted Device Distribution
Predicted_Device
UNKNOWN    1.0
Name: proportion, dtype: float64


/projects/envs/conda/jzander/envs/python_notebook/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but RadiusNeighborsClassifier was fitted with feature names
  warnings.warn(
/projects/envs/conda/jzander/envs/python_notebook/lib/python3.14/site-packages/sklearn/neighbors/_classification.py:868: UserWarning: Outlier label UNKNOWN is not in training classes. All class probabilities of outliers will be assigned with 0.
  warnings.warn(
/projects/envs/conda/jzander/envs/python_notebook/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but RadiusNeighborsClassifier was fitted with feature names
  warnings.warn(
/projects/envs/conda/jzander/envs/python_notebook/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but RadiusNeighborsClassifier was fitted with feature names
  warnings.warn(
/projects/envs/con

Predict NA samples

In [30]:
df_na = pd.read_csv("EKP_nan_parsed_cleaned.csv")

print("NA Results:")
predict(df_na)

NA Results:
1862


/projects/envs/conda/jzander/envs/python_notebook/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but RadiusNeighborsClassifier was fitted with feature names
  warnings.warn(
/projects/envs/conda/jzander/envs/python_notebook/lib/python3.14/site-packages/sklearn/neighbors/_classification.py:868: UserWarning: Outlier label UNKNOWN is not in training classes. All class probabilities of outliers will be assigned with 0.
  warnings.warn(
/projects/envs/conda/jzander/envs/python_notebook/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but RadiusNeighborsClassifier was fitted with feature names
  warnings.warn(



Predicted Device Distribution
Predicted_Device
UNKNOWN    0.995166
VITEK      0.004834
Name: proportion, dtype: float64
